# Brain Tumor Classification
**Entrenamiento en Google Colab**

---
### ✅ Solo tienes que cambiar 1 cosa:
En la **Celda 3** cambia `MODELO` por el modelo que te tocó entrenar.

## Paso 1: Montar Google Drive

Ejecuta esta celda y **permite el acceso** cuando Google lo pida.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Paso 2: Configurar modelo y rutas

✏️ **Cambia `MODELO`** por el que te tocó:
- `"resnet50"`
- `"efficientnet_b0"`
- `"vit"`
- `"swin"`
- `"cvt"`

📁 **`RUTA_DRIVE`** es donde subiste la carpeta `BrainTumor/` a tu Drive.

📁 **`RUTA_DATASET`** es donde subiste el dataset a tu Drive.

In [ ]:
# ========================================
# TU SOLO CAMBIA ESTO:
# ========================================
MODELO = "resnet50"  # ← cambia por tu modelo

# Rutas en tu Google Drive (ajústalas si es necesario)
RUTA_DRIVE = "/content/drive/MyDrive/BrainTumor"
RUTA_DATASET = "/content/drive/MyDrive/BrainTumor_Dataset"
# ========================================

print(f"Modelo a entrenar: {MODELO}")
print(f"Proyecto en:      {RUTA_DRIVE}")
print(f"Dataset en:       {RUTA_DATASET}")

## Paso 3: Copiar proyecto y dataset

Esto copia los archivos a Colab para que sea más rápido.

In [ ]:
import os, shutil

# Copiar proyecto
if os.path.exists("/content/BrainTumor"):
    shutil.rmtree("/content/BrainTumor")
shutil.copytree(RUTA_DRIVE, "/content/BrainTumor")
os.chdir("/content/BrainTumor")
print("Proyecto copiado ✅")

# Copiar dataset dentro del proyecto
dst = "/content/BrainTumor/dataset/BrainTumor_Dataset"
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(RUTA_DATASET, dst)
print("Dataset copiado ✅")

print(f"\nContenido de dataset/: {os.listdir('dataset')}")

## Paso 4: Instalar dependencias

Solo la primera vez. Colab ya trae PyTorch, pero instalamos lo que falta.

In [ ]:
!pip install -q matplotlib scikit-learn tqdm

# Si tu modelo es 'cvt', instala también:
if MODELO == "cvt":
    !pip install -q cvt-pytorch

print("Dependencias listas ✅")

## Paso 5: Verificar GPU

Tiene que salir **`True`** y mostrar el nombre de la GPU (ej. Tesla T4).

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No hay GPU. El entrenamiento será muy lento.")
    print("   Ve a: Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU")

---
## ⚡ Prueba rápida (2 épocas)

Ejecuta esto para verificar que todo funciona antes del entrenamiento completo.
Tarda ~2 minutos.

In [ ]:
!python src/train.py --model $MODELO --epochs 2
print("\n✅ Prueba exitosa. Listo para entrenamiento completo.")

---
## 🚀 Entrenamiento completo (30 épocas)

⚠️ **Esto se demora 2-3 horas**. No cierres la pestaña.

Si ya hiciste la prueba rápida arriba y funcionó, ejecuta esta celda.

In [ ]:
!python src/train.py --model $MODELO
print("\n✅ Entrenamiento completado.")

---
## 📊 Evaluar modelo

Esto genera las métricas finales y las gráficas.
Ejecuta **después** del entrenamiento completo.

In [ ]:
!python src/evaluate.py --model $MODELO
print("\n✅ Evaluación completada.")

---
## 💾 Guardar resultados en Drive

Esto copia los resultados a tu Drive para que no los pierdas al cerrar Colab.

In [ ]:
import shutil

dst = f"{RUTA_DRIVE}/results/{MODELO}"
os.makedirs(dst, exist_ok=True)
shutil.copytree(f"/content/BrainTumor/results/{MODELO}", dst, dirs_exist_ok=True)
print(f"Resultados guardados en: {dst}")
print("✅ Ya puedes cerrar Colab.")

---
## 👀 Ver resultados

Ejecuta esto para ver las métricas y gráficas directamente en Colab.

In [ ]:
import json
from pathlib import Path

ruta = Path(f"results/{MODELO}")

if (ruta / "metrics.json").exists():
    with open(ruta / "metrics.json") as f:
        metrics = json.load(f)
    print(f"Métricas de {MODELO}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
else:
    print("Todavía no hay métricas. Ejecuta primero Entrenamiento + Evaluación.")

# Mostrar gráficas
from IPython.display import Image, display
for img in ["confusion_matrix.png", "roc_curve.png"]:
    path = ruta / img
    if path.exists():
        display(Image(filename=str(path)))